# 03 · Validation and guardrails

**Role of this layer.** "The LLM proposes, the system disposes." A plan is inert
JSON; the validator decides, *without touching any data values*, whether it may run:

* **BLOCK**: cannot run (unknown capability, bad parameter, returns of a yield...).
* **APPROVAL**: runs only after a human explicitly approves (look-ahead, material assumptions).
* **WARN**: runs; the warning is shown and passed to the narrator.

It does a metadata *dry run*: walking the steps and propagating data type, kind
and frequency through the registry, which is how it catches "returns of a yield"
or "the spread of a daily and a weekly series" before anything executes.

Below, every static guardrail G01-G13 is triggered on a deliberately broken plan
(G11 was retired together with the statistical tests; codes are never reused),
and the last section edits a plan's parameters to show the same validator
re-checking every change (G10 appearing, the G14 warning, an out-of-range BLOCK).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
import copy

from quantlab.data import load_catalog
from quantlab.plan import Plan
from quantlab.scenarios import SCENARIOS_BY_ID
from quantlab.validation import GUARDRAILS, check_schema, fill_defaults, validate_plan

catalog = load_catalog(settings.catalog_path)


def scenario_plan(scenario_id):
    """A mutable copy of a demo scenario's canned plan."""
    s = SCENARIOS_BY_ID[scenario_id]
    return {"plan_id": "demo", "request": s.request, **copy.deepcopy(s.plan)}


# A clean rates plan (not one of the example workflows): the 10y-2y spread and its summary.
SPREAD = {
    "dataset_ids": ["DGS10", "DGS2"],
    "steps": [
        {"id": "s1", "capability": "load_series", "params": {"dataset_id": "DGS10"}},
        {"id": "s2", "capability": "load_series", "params": {"dataset_id": "DGS2"}},
        {"id": "s3", "capability": "spread", "inputs": {"a": "s1", "b": "s2"}},
        {"id": "s4", "capability": "summary_stats", "inputs": {"series": "s3"}},
    ],
}


def spread_plan():
    """A mutable copy of the spread plan."""
    return {"plan_id": "demo", "request": "10y-2y spread", **copy.deepcopy(SPREAD)}


def codes_for(raw, **budgets):
    """Parse and validate a plan dict; return the issue codes raised."""
    plan, schema_issues = check_schema(raw)
    if plan is None:
        return sorted({i.code for i in schema_issues})
    return validate_plan(fill_defaults(plan), catalog, **budgets).codes()


pd.DataFrame(
    [
        {
            "code": g.code,
            "name": g.name,
            "severity": g.severity.value,
            "stage": g.stage,
            "what goes wrong without it": g.risk,
        }
        for g in GUARDRAILS.values()
        if g.stage == "static"
    ]
)

,code,name,severity,stage,what goes wrong without it
0,G01,SCHEMA_INVALID,BLOCK,static,"Malformed or partial planner output would have to be guessed at, and extra fields (e.g. 'code') could slip..."
1,G02,UNKNOWN_CAPABILITY,BLOCK,static,"The LLM could ask for an analysis nobody wrote or tested (e.g. a forecasting model, or 'run Python')."
2,G03,INVALID_PARAM,BLOCK,static,"Typos, wrong types or extreme values (window=1, quantile=1.5) would crash or silently produce nonsense."
3,G04,BAD_REFERENCE,BLOCK,static,"Steps could read missing outputs or outputs of later steps, leaving the order of computation undefined."
4,G05,TYPE_MISMATCH,BLOCK,static,"Meaningless chains would run, e.g. volatility of a price level or summary statistics of regime flags."
5,G06,BUDGET_EXCEEDED,BLOCK,static,Runaway plans could use unbounded compute and become impossible for a human to review.
6,G07,UNKNOWN_DATASET,BLOCK,static,The LLM could invent a dataset id or reach for data outside the approved catalog.
7,G08,RETURNS_ON_RATES,BLOCK,static,'Returns' of a yield are meaningless (1% -> 2% is not +100%); rates must be differenced.
8,G09,FREQUENCY_MISMATCH,BLOCK,static,Combining a daily with a weekly series silently keeps only the dates they share and mixes frequencies.
9,G10,LOOK_AHEAD,APPROVAL,static,"Full-sample thresholds use future data, so results look better than anything achievable in real time."


A clean plan raises nothing; scenario 1 only needs the return-method approval.

In [3]:
print("spread plan:", codes_for(spread_plan()))
print("scenario 1:", codes_for(scenario_plan("volatility_flag_success")))

spread plan: []
scenario 1: ['G12']


Now one crafted plan per guardrail:

In [4]:
S1 = "volatility_flag_success"
cases = {}

raw = scenario_plan(S1)
raw["steps"][0]["code"] = "import os"
cases["G01 extra field 'code'"] = raw

raw = scenario_plan(S1)
raw["steps"].append({"id": "s6", "capability": "sarima", "inputs": {"series": "s3"}})
cases["G02 a planner inventing a SARIMA model"] = raw

raw = scenario_plan(S1)
raw["steps"][3]["params"] = {"window": 2}
cases["G03 window below range"] = raw

raw = scenario_plan(S1)
raw["steps"][3]["inputs"] = {"series": "s5"}
cases["G04 reference to a later step"] = raw

raw = scenario_plan(S1)
raw["steps"][3]["inputs"] = {"series": "s2"}
cases["G05 volatility of a price level"] = raw

raw = spread_plan()
raw["steps"] += [
    {"id": f"s{i}", "capability": "summary_stats", "inputs": {"series": "s3"}} for i in range(5, 17)
]
cases["G06 16 steps > budget of 15"] = raw

raw = spread_plan()
raw["steps"][1]["params"] = {"dataset_id": "BITCOIN"}
cases["G07 dataset not in catalog"] = raw

raw = spread_plan()
raw["steps"] = [
    raw["steps"][0],
    {
        "id": "s2",
        "capability": "compute_returns",
        "inputs": {"series": "s1"},
        "params": {"method": "simple"},
    },
]
cases["G08 returns of a yield"] = raw

cases["G09 daily VIX flags vs weekly returns (scenario 5)"] = scenario_plan(
    "frequency_mismatch_block"
)
cases["G10 full-sample threshold (scenario 2)"] = scenario_plan("volatility_flag_lookahead")
cases["G12 return method needs approval (scenario 1)"] = scenario_plan(S1)
cases["G13 planner refusal (scenario 3)"] = scenario_plan("sarima_forecast_block")
cases["G13 prompt injection refused (scenario 4)"] = scenario_plan("prompt_injection_block")

pd.DataFrame(
    [{"case": name, "codes raised": ", ".join(codes_for(raw))} for name, raw in cases.items()]
)

,case,codes raised
0,G01 extra field 'code',G01
1,G02 a planner inventing a SARIMA model,"G02, G12"
2,G03 window below range,"G03, G12"
3,G04 reference to a later step,"G04, G12"
4,G05 volatility of a price level,"G05, G12"
5,G06 16 steps > budget of 15,G06
6,G07 dataset not in catalog,G07
7,G08 returns of a yield,"G08, G12"
8,G09 daily VIX flags vs weekly returns (scenario 5),"G09, G12"
9,G10 full-sample threshold (scenario 2),"G10, G12"


Each issue carries a message and a suggestion. For example, scenario 5 (daily VIX
flags compared with weekly returns):

In [5]:
report = validate_plan(fill_defaults(Plan(**scenario_plan("frequency_mismatch_block"))), catalog)
for issue in report.issues:
    print(f"[{issue.severity}] {issue.issue_id}: {issue.message}\n    -> {issue.suggestion}")

[APPROVAL] G12:s5:return_method: The return method is a material modelling choice: simple returns compound multiplicatively, log returns add up over time, and the two give different averages. Chosen: method='simple'.
    -> Approve if this choice is intended, or edit the parameter.
[BLOCK] G09:s6: compare_groups inputs have different frequencies: 'values' is weekly, 'flags' is daily.
    -> Resample VIXCLS to weekly right after load_series (resample(freq='W')), before computing the 'flags' input.


The dry run: predicted output metadata of each step of scenario 1 (no data touched).

In [6]:
report = validate_plan(fill_defaults(Plan(**scenario_plan(S1))), catalog)
pd.DataFrame(report.dry_run).T[["data_type", "kind", "frequency", "return_method", "dataset_ids"]]

,data_type,kind,frequency,return_method,dataset_ids
s1,series,price_level,D,None,[NASDAQCOM]
s2,series,price_level,W,None,[NASDAQCOM]
s3,series,returns,W,simple,[NASDAQCOM]
s4,series,volatility,W,simple,[NASDAQCOM]
s5,flags,flags,W,simple,[NASDAQCOM]


### Editing a plan: every change goes through the same validator

A user may change parameter VALUES only, through the engine's `apply_param_edits`.
Each edit is applied to a copy of the plan and the full validator runs again, so
issues appear and disappear; any edit also resets the approvals.

In [7]:
from quantlab.engine import Session

s1 = SCENARIOS_BY_ID[S1]
session = Session.start(s1.request, list(s1.dataset_ids), "mock", settings)
print("as proposed:          ", session.create_plan().codes())
print(
    "method -> full_sample:", session.apply_param_edits({"s5": {"method": "full_sample"}}).codes()
)
print("method -> expanding:  ", session.apply_param_edits({"s5": {"method": "expanding"}}).codes())

as proposed:           ['G12']
method -> full_sample: ['G10', 'G12']
method -> expanding:   ['G12']


Switching the resample step from weekly to daily leaves the windows in the wrong units (G14):

In [8]:
report = session.apply_param_edits({"s2": {"freq": "D"}})
for issue in report.warnings:
    print(f"[{issue.severity}] {issue.code}: {issue.message}\n    -> {issue.suggestion}")

[WARN] G14: Frequency changed from weekly to daily: window 12 now means 12 trading days, not 12 weeks.
    -> Consider 60 for roughly 12 weeks.
[WARN] G14: Frequency changed from weekly to daily: min_periods 52 now means 52 trading days, not 52 weeks.
    -> Consider 260 for roughly 52 weeks.


The widgets in the app are bounded, but the validator is the backstop: an
out-of-range value is a BLOCK, and a structural edit is rejected outright.

In [9]:
for edits in [{"s5": {"quantile": 1.5}}, {"s4": {"capability": "sarima"}}]:
    session.reset_to_proposal()  # start each example from the planner's proposal
    report = session.apply_param_edits(edits)
    for issue in report.blocks:
        print(f"{edits} -> [{issue.severity}] {issue.code}: {issue.message}")
print("after a reset:", session.reset_to_proposal().codes())

{'s5': {'quantile': 1.5}} -> [BLOCK] G03: 'quantile'=1.5 is outside [0.5, 0.99].
{'s4': {'capability': 'sarima'}} -> [BLOCK] G03: Edit rejected: 'capability' is not a parameter of s4 (rolling_volatility).
after a reset: ['G12']
